In [0]:
import pyspark.sql.functions as F

In [0]:
input_path = 's3://ims-dbr-raw/yellow_trip_records/'
output_path = 's3://ims-dbr-stage/yellow_trip_records/'

In [0]:
df = spark.read.parquet(input_path)
display(df)

#Cast and _derive_ columns

In [0]:
df_stage = df \
       .withColumn('VendorID', F.col('VendorID').cast('int')) \
       .withColumn('tpep_pickup_datetime', F.col('tpep_pickup_datetime').cast('timestamp')) \
       .withColumn('tpep_dropoff_datetime', F.col('tpep_dropoff_datetime').cast('timestamp')) \
       .withColumn('passenger_count', F.col('passenger_count').cast('int')) \
       .withColumn('trip_distance', F.col('trip_distance').cast('double')) \
       .withColumn('RatecodeID', F.col('RatecodeID').cast('int')) \
       .withColumn('store_and_fwd_flag', F.col('store_and_fwd_flag').cast('string')) \
       .withColumn('PULocationID', F.col('PULocationID').cast('int')) \
       .withColumn('DOLocationID', F.col('DOLocationID').cast('int')) \
       .withColumn('payment_type', F.col('payment_type').cast('int')) \
       .withColumn('fare_amount', F.col('fare_amount').cast('double')) \
       .withColumn('extra', F.col('extra').cast('double')) \
       .withColumn('mta_tax', F.col('mta_tax').cast('double')) \
       .withColumn('tip_amount', F.col('tip_amount').cast('double')) \
       .withColumn('tolls_amount', F.col('tolls_amount').cast('double')) \
       .withColumn('improvement_surcharge', F.col('improvement_surcharge').cast('double')) \
       .withColumn('total_amount', F.col('total_amount').cast('double')) \
       .withColumn('congestion_surcharge', F.col('congestion_surcharge').cast('double')) \
       .withColumn('airport_fee', F.col('airport_fee').cast('double'))\
       .withColumn('trip_duration', 
                   F.timestamp_diff('minute', F.col('tpep_pickup_datetime'), F.col('tpep_dropoff_datetime'))) \
       .withColumn('pu_hour_of_day', F.hour('tpep_pickup_datetime')) \
       .withColumn('pu_day_of_week', F.dayofweek('tpep_pickup_datetime')) \
       .withColumn('pu_year', F.year('tpep_pickup_datetime')) \
       .withColumn('pu_month', F.month('tpep_pickup_datetime')) \
       .withColumn('do_hour_of_day', F.hour('tpep_dropoff_datetime')) \
       .withColumn('do_day_of_week', F.dayofweek('tpep_dropoff_datetime')) \
       .withColumn('do_year', F.year('tpep_dropoff_datetime')) \
       .withColumn('do_month', F.month('tpep_dropoff_datetime')) \
      \
       .withColumn('is_pu_unknown', (F.col('PULocationID') == 264) | (F.col('PULocationID') == 265)) \
       .withColumn('is_do_unknown', (F.col('DOLocationID') == 264) | (F.col('DOLocationID') == 265)) \
       .fillna({
          'passenger_count': 1,
          'RatecodeID': 1,
          'store_and_fwd_flag': 'Unknown',
          'airport_fee': 0,
          'congestion_surcharge': 0}) \
       .filter(
          (F.col('fare_amount') >0 ) & 
          (F.col('passenger_count')>0) & 
          (F.col('trip_distance')>0) & 
          (F.col('trip_duration')>0)) \
       .dropDuplicates(subset = ['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'PULocationID', 'DOLocationID'])            
       
           

In [0]:
## Write to Stage Table
df_stage.write \
  .mode("overwrite") \
  .parquet(output_path)

In [0]:
%sql
CREATE TABLE IF NOT EXISTS nytaxi.staging.yellow_trip_records
USING PARQUET
LOCATION 's3://ims-dbr-stage/yellow_trip_records/';